In [1]:
import tensorflow as tf
import numpy as np
import numpy.typing as npt
import matplotlib.pyplot as plt
import nibabel as nib

In [ ]:
TRAINING_DATA_PATH = "training-data/{}/BraTS20_Training_{:03}{}.nii"

In [4]:
def get_paths_from_id(id: int):
    """Get path to inputs volume and segmented volume by case ID.

    Args:
        id (int): ID of the volume, ranging from 1-369.

    Returns:
        tuple[str,str]: A tuple with the inputs path first and the segmentation path second
    """    
    return TRAINING_DATA_PATH.format('input', id, ''), TRAINING_DATA_PATH.format('seg', id, '_seg')

def load_nii(path: str) -> npt.NDArray[np.float32]:
    """Load a Nifti file (as `float32`)

    Args:
        path (str): Path to the Nifti file

    Returns:
        npt.NDArray[np.float32]: The loaded `nibabel` object
    """    
    return nib.load(path).get_fdata().astype(np.float32)

def load_volume(id: int) -> tuple[npt.NDArray[np.float32], npt.NDArray[np.float32]]:
    """Load a volume by case ID

    Args:
        id (int): The case ID of the volume

    Returns:
        tuple[npt.NDArray[np.float32], npt.NDArray[np.float32]]: The loaded volume's inputs (t1, t1ce, t2 as channels), and the segmentation mask
    """    
    inputs_path, seg_path = get_paths_from_id(id)    
    x = load_nii(inputs_path)
    y = load_nii(seg_path) # (H, W, D)
    return x, y

def slice_generator(volume_ids: npt.ArrayLike):
    """Generate slices from volumes with the given IDs

    Args:
        volume_ids (npt.ArrayLike): List of case IDs (integers 1-369)

    Yields:
        tuple[npt.NDArray[np.float32],npt.NDArray[np.float32]]: The loaded volumes' inputs (t1, t1ce, t2 as channels) first, and the segmentation masks second
    """    
    for id in np.asarray(volume_ids):
        x, y = load_volume(id)  # (H,W,D,4), pre-preprocessed
        for z in range(x.shape[2]):
            yield id, z, x[:, :, z, 1:], y[:, :, z]

In [ ]:
def plot_slice(input: npt.NDArray[np.float32], seg: npt.NDArray[np.float32], title: str=None):
    """Plot the modalities of a slice from a volume.

    Args:
        input (npt.NDArray[np.float32]): The input modalities, in numpy form.
        seg (npt.NDArray[np.float32]): The segmentation to plot alongside the input.
        title (str, optional): The title of the plot. Defaults to None.
    """    
    orig = input.copy()
    input -= input.min(axis=(0,1))
    input /= input.max(axis=(0,1))
    input[orig == 0.0] = 0.0
    
    fig, (ax1, ax2) = plt.subplots(1,2)
    ax1.imshow(input)
    ax2.imshow(seg)
    ax1.scatter([], [], label="T1", c='r')
    ax1.scatter([], [], label="T1ce", c='g')
    ax1.scatter([], [], label="T2", c='b')
    fig.legend(loc=[0.03,0.13])
    ax1.axis("off")
    ax2.axis("off")
    fig.suptitle(title, y=0.1, verticalalignment="bottom")
    fig.tight_layout()

In [8]:
dataset = tf.data.Dataset.from_generator(
    lambda: slice_generator(list(range(1, 370))),
    output_signature=(
        tf.TensorSpec(shape=(), dtype=tf.uint16),
        tf.TensorSpec(shape=(), dtype=tf.uint16),
        tf.TensorSpec(shape=(128, 128, 3), dtype=np.float32),
        tf.TensorSpec(shape=(128, 128), dtype=np.float32)
    )
)
dataset = dataset.shuffle(buffer_size=2048).cache().batch(32)